# L12c: Long Short-Term Memory (LSTM) Networks

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Understand LSTM gating architecture and cell state:__ Explain how the forget, input, and output gates control information flow through the cell state, and why this addresses the vanishing gradient problem.
> * __Apply LSTM equations to compute outputs and count parameters:__ Write the gate equations, trace the cell state update for a given input sequence, and compute the total parameter count for a given architecture.
> * __Compare LSTM and Elman RNN capabilities:__ Identify when an LSTM is preferred over an Elman RNN based on sequence length, temporal dependency range, and parameter budget.

### Example
This lecture is accompanied by [a numerical example on LSTM networks for fed-batch CHO bioreactor prediction](CHEME-5820-L12c-Example-LSTM-CHO-Spring-2026.ipynb), where we apply the LSTM architecture to predict multi-state bioreactor trajectories.
___

## Motivation: Why Gating?
In the [L12a lecture on Recurrent Neural Networks](CHEME-5820-L12a-Lecture-RecurrentNetworks-Spring-2026.ipynb), we saw that Elman RNNs suffer from vanishing and exploding gradients when trained with Backpropagation Through Time (BPTT). The gradient of the loss with respect to early hidden states involves a product of Jacobian matrices across all time steps. When the dominant eigenvalue of the recurrent weight matrix $\mathbf{U}_h$ satisfies $|\lambda| < 1$, gradients decay exponentially; when $|\lambda| > 1$, they grow exponentially.

> __The cell state as a gradient highway__
>
> The LSTM solves this problem by introducing a *cell state* $\mathbf{c}_t$ that flows through the network with minimal transformation. The cell state update is:
> $$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t$$
> where $\mathbf{f}_t$ is the forget gate and $\mathbf{i}_t$ is the input gate (both in $(0,1)$). When $\mathbf{f}_t \approx 1$ and $\mathbf{i}_t \approx 0$, the cell state passes unchanged: $\mathbf{c}_t \approx \mathbf{c}_{t-1}$. The gradient $\partial \mathbf{c}_t / \partial \mathbf{c}_{t-1} = \text{diag}(\mathbf{f}_t)$, which is close to the identity matrix when the forget gate is near 1. This provides an unimpeded path for gradients to flow backward through many time steps.

The LSTM was introduced by [Hochreiter and Schmidhuber (1997)](https://doi.org/10.1162/neco.1997.9.8.1735) and has become the standard architecture for sequence modeling tasks requiring long-range temporal dependencies.
___

## LSTM: Mathematical Formulation
The LSTM cell computes six quantities at each time step $t$.

> __LSTM gate equations__
>
> Given input $\mathbf{x}_t \in \mathbb{R}^{d_{\text{in}}}$, previous hidden state $\mathbf{h}_{t-1} \in \mathbb{R}^{h}$, and previous cell state $\mathbf{c}_{t-1} \in \mathbb{R}^{h}$:
>
> $$
\begin{align*}
\mathbf{f}_t &= \sigma(\mathbf{W}_f \mathbf{x}_t + \mathbf{U}_f \mathbf{h}_{t-1} + \mathbf{b}_f) & \text{(forget gate)} \\
\mathbf{i}_t &= \sigma(\mathbf{W}_i \mathbf{x}_t + \mathbf{U}_i \mathbf{h}_{t-1} + \mathbf{b}_i) & \text{(input gate)} \\
\tilde{\mathbf{c}}_t &= \tanh(\mathbf{W}_c \mathbf{x}_t + \mathbf{U}_c \mathbf{h}_{t-1} + \mathbf{b}_c) & \text{(candidate cell state)} \\
\mathbf{c}_t &= \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t & \text{(cell state update)} \\
\mathbf{o}_t &= \sigma(\mathbf{W}_o \mathbf{x}_t + \mathbf{U}_o \mathbf{h}_{t-1} + \mathbf{b}_o) & \text{(output gate)} \\
\mathbf{h}_t &= \mathbf{o}_t \odot \tanh(\mathbf{c}_t) & \text{(hidden state)}
\end{align*}
> $$
>
> where $\sigma(\cdot)$ is the sigmoid function, $\odot$ denotes element-wise multiplication, and the weight matrices and bias vectors are:
> * $\mathbf{W}_f, \mathbf{W}_i, \mathbf{W}_c, \mathbf{W}_o \in \mathbb{R}^{h \times d_{\text{in}}}$ (input weights for each gate)
> * $\mathbf{U}_f, \mathbf{U}_i, \mathbf{U}_c, \mathbf{U}_o \in \mathbb{R}^{h \times h}$ (recurrent weights for each gate)
> * $\mathbf{b}_f, \mathbf{b}_i, \mathbf{b}_c, \mathbf{b}_o \in \mathbb{R}^{h}$ (biases for each gate)

The output projection maps the hidden state to the prediction:
$$\mathbf{y}_t = \sigma_y(\mathbf{V}_y \mathbf{h}_t + \mathbf{b}_y)$$
where $\mathbf{V}_y \in \mathbb{R}^{d_{\text{out}} \times h}$, $\mathbf{b}_y \in \mathbb{R}^{d_{\text{out}}}$, and $\sigma_y$ is the output activation (sigmoid when targets are in $[0,1]$, linear for unbounded regression).

Each gate serves a specific role:
* **Forget gate** $\mathbf{f}_t$: controls what information to discard from the previous cell state.
* **Input gate** $\mathbf{i}_t$: controls what new information to write to the cell state.
* **Output gate** $\mathbf{o}_t$: controls what information from the cell state to expose as the hidden state.
___

## Parameter Counting
The LSTM has four gates, each with its own input weight matrix ($\mathbf{W}$), recurrent weight matrix ($\mathbf{U}$), and bias vector ($\mathbf{b}$), plus an output projection layer.

> __LSTM parameter count__
>
> Each gate contributes $h \cdot d_{\text{in}} + h \cdot h + h = h(d_{\text{in}} + h + 1)$ parameters. With four gates and an output layer:
> $$N_{\text{total}} = 4h(h + d_{\text{in}} + 1) + d_{\text{out}}(h + 1)$$

### Numerical Example
For the CHO bioreactor LSTM with $d_{\text{in}} = 10$, $h = 128$, $d_{\text{out}} = 7$:

$$N_{\text{gates}} = 4 \times 128 \times (128 + 10 + 1) = 4 \times 128 \times 139 = 71{,}168$$
$$N_{\text{output}} = 7 \times (128 + 1) = 903$$
$$N_{\text{total}} = 71{,}168 + 903 = 72{,}071$$

### Comparison with Elman RNN
An Elman RNN with the same dimensions has:
$$N_{\text{Elman}} = h(h + d_{\text{in}} + 1) + d_{\text{out}}(h + 1) = 128 \times 139 + 903 = 17{,}792 + 903 = 18{,}695$$

The LSTM uses approximately $4\times$ more parameters than the Elman RNN because it has four gate computations instead of one hidden state computation. This increased capacity enables the LSTM to learn what to remember and what to forget, at the cost of more computation per time step.
___

## Gated Recurrent Unit (GRU)
The Gated Recurrent Unit (GRU), introduced by [Cho et al. (2014)](https://arxiv.org/abs/1406.1078), is a simplified gating architecture that combines the forget and input gates into a single *update gate* and merges the cell state and hidden state.

> __GRU equations__
>
> $$
\begin{align*}
\mathbf{z}_t &= \sigma(\mathbf{W}_z \mathbf{x}_t + \mathbf{U}_z \mathbf{h}_{t-1} + \mathbf{b}_z) & \text{(update gate)} \\
\mathbf{r}_t &= \sigma(\mathbf{W}_r \mathbf{x}_t + \mathbf{U}_r \mathbf{h}_{t-1} + \mathbf{b}_r) & \text{(reset gate)} \\
\tilde{\mathbf{h}}_t &= \tanh(\mathbf{W}_h \mathbf{x}_t + \mathbf{U}_h (\mathbf{r}_t \odot \mathbf{h}_{t-1}) + \mathbf{b}_h) & \text{(candidate hidden state)} \\
\mathbf{h}_t &= (1 - \mathbf{z}_t) \odot \mathbf{h}_{t-1} + \mathbf{z}_t \odot \tilde{\mathbf{h}}_t & \text{(hidden state update)}
\end{align*}
> $$
>
> The GRU has 3 gate weight sets instead of 4, giving a parameter count of $3h(h + d_{\text{in}} + 1) + d_{\text{out}}(h + 1)$. For the same dimensions as above: $3 \times 128 \times 139 + 903 = 54{,}279$.

In practice, LSTMs and GRUs achieve comparable performance on most tasks. GRUs are slightly more parameter-efficient and faster to train, while LSTMs provide more flexibility through the separate cell state. The choice often comes down to empirical performance on the specific task.
___

## Summary
In this lecture, we introduced the LSTM architecture as a solution to the vanishing gradient problem in Elman RNNs, derived the gate equations and parameter count, and compared the LSTM to the GRU.

> __Key Takeaways:__
>
> * **The cell state provides unimpeded gradient flow:** The LSTM cell state acts as a gradient highway, allowing information to persist across many time steps without exponential decay or growth of gradients.
> * **Gating mechanisms learn what to remember and forget:** The forget, input, and output gates use learned sigmoid activations to selectively control information flow, enabling the network to maintain relevant long-range context while discarding irrelevant information.
> * **LSTMs use approximately 4x more parameters than Elman RNNs:** The four gate computations increase the parameter count by a factor of four relative to an Elman RNN with the same hidden dimension, trading parameter efficiency for the ability to learn long-range temporal dependencies.

For a numerical example applying LSTM networks to a fed-batch bioreactor prediction task, see the [L12c example notebook](CHEME-5820-L12c-Example-LSTM-CHO-Spring-2026.ipynb).
___